In [1]:
import os
import pandas as pd
from src import config, ingestion
from pathlib import Path

In [2]:
os.makedirs(config.DATA_RAW_DIR, exist_ok=True)

In [3]:
historical_weather = ingestion.fetch_all_zones(
    start_date=config.HISTORICAL_START,
    end_date=config.HISTORICAL_END,
    mode="historical"
)

2026-04-25 16:39:08,782 - INFO - Fetching hourly weather for High Relief (2020-01-01 to 2026-04-20)
2026-04-25 16:39:11,500 - INFO - Fetching hourly weather for Low Relief (2020-01-01 to 2026-04-20)
2026-04-25 16:39:13,545 - INFO - Fetching hourly weather for Moderate Relief (2020-01-01 to 2026-04-20)


In [4]:
for zone, df in historical_weather.items():
    df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_weather_historical.csv", index=False)

In [5]:
historical_flood = ingestion.fetch_all_zones(
    start_date=config.HISTORICAL_START,
    end_date=config.HISTORICAL_END,
    mode="flood"
)

2026-04-25 16:39:16,421 - INFO - Fetching flood data for High Relief (2020-01-01 to 2026-04-20)
2026-04-25 16:39:17,065 - INFO - Fetching flood data for Low Relief (2020-01-01 to 2026-04-20)
2026-04-25 16:39:17,591 - INFO - Fetching flood data for Moderate Relief (2020-01-01 to 2026-04-20)


In [6]:
for zone, df in historical_flood.items():
    if not df.empty:
        df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_flood_historical.csv", index=False)

In [7]:
forecast_weather = ingestion.fetch_all_zones(
    start_date=None, 
    end_date=None, 
    mode="forecast"
)

2026-04-25 16:39:18,245 - INFO - Fetching 15-day forecast for High Relief
2026-04-25 16:39:18,726 - INFO - Fetching 15-day forecast for Low Relief
2026-04-25 16:39:19,225 - INFO - Fetching 15-day forecast for Moderate Relief


In [8]:
for zone, df in forecast_weather.items():
    df.to_csv(config.DATA_RAW_DIR / f"{zone.lower()}_weather_forecast.csv", index=False)

In [9]:
def audit_data(df, name):
    expected = pd.date_range(start=df['time'].min(), end=df['time'].max())
    gaps = expected.difference(df['time'])
    nulls = df.isnull().sum()
    return {
        "Dataset": name,
        "Rows": len(df),
        "Start": df['time'].min(),
        "End": df['time'].max(),
        "Gaps": len(gaps),
        "Nulls": nulls[nulls > 0].to_dict()
    }

In [10]:
results = []
for z, df in historical_weather.items():
    results.append(audit_data(df, f"{z} Weather"))
for z, df in historical_flood.items():
    if not df.empty:
        results.append(audit_data(df, f"{z} Flood"))

pd.DataFrame(results)

,Dataset,Rows,Start,End,Gaps,Nulls
0,High Relief Weather,55248,2020-01-01,2026-04-20 23:00:00,0,{}
1,Low Relief Weather,55248,2020-01-01,2026-04-20 23:00:00,0,{}
2,Moderate Relief Weather,55248,2020-01-01,2026-04-20 23:00:00,0,{}
3,High Relief Flood,2302,2020-01-01,2026-04-20 00:00:00,0,{}
4,Low Relief Flood,2302,2020-01-01,2026-04-20 00:00:00,0,{}
5,Moderate Relief Flood,2302,2020-01-01,2026-04-20 00:00:00,0,{}
